# Experiment 08: Minimum Frequency Gap Sweep

Vary the minimum theta separation and track basis conditioning plus alignment degradation.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for label, results in result_map.items():
        row = overall_row(results).copy()
        row["run_label"] = label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is None:
        return df
    keep = [col for col in ["run_label", *columns] if col in df.columns]
    return df[keep]


In [ ]:
BASE_CONFIG = dict(
    time_mode="discrete",
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    RANDOM_AMPLITUDE=False,
    RANDOM_PHASE=True,
    USE_NOISE=False,
    HIDDEN_DIM=128,
    BOTTLENECK_MULTIPLIER=4,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [ ]:
results = {}
for num_freqs in [4, 8]:
    seq_len = 4096 if num_freqs == 4 else 8192
    for min_gap in [0.20, 0.12, 0.08, 0.04, 0.02]:
        if min_gap * (num_freqs - 1) >= 0.80:
            print(f"skip infeasible setting: k={num_freqs}, min_gap={min_gap:.02f}pi")
            continue
        for model_id, lr in [("AN003_LINEAR", 1e-2), ("AN002_NO_BN_TANH", 1e-3)]:
            cfg = ExperimentConfig(
                **BASE_CONFIG,
                SEQ_LEN=seq_len,
                NUM_FREQS=num_freqs,
                LAG=4 * num_freqs,
                MIN_DELTA_THETA=min_gap * np.pi,
                MODEL_ID=model_id,
                LR=lr,
                EPOCHS=1500,
            )
            label = f"model={model_id},k={num_freqs},min_gap={min_gap:.02f}pi"
            results[label] = run_experiment(cfg)

collect_overall_rows(
    results,
    columns=[
        "min_delta_theta",
        "fourier_min_singular_value_mean",
        "fourier_condition_number_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
    ],
)
